# Обучающий пайплайн — `uavtrain`

Весь путь по секциям: **скачка датасетов → подготовка → обучение (YOLOv8 + lightweight CNN) → eval → export весов в `../models/`**.

Каждая секция вызывает функции пакета `uavtrain` (`train/src/uavtrain/`). Тяжёлые шаги (обучение) рассчитаны на сервер с GPU (RTX 3090); подготовка/eval — везде. Часть датасетов требует ручной загрузки (см. вывод `list-datasets` и `manual_instructions`).

Запуск как альтернатива ноутбуку — CLI: `python -m uavtrain.cli <команда>` (см. `uavtrain/cli.py`).

## 0. Подготовка окружения

In [ ]:
# из train/: pip install -e .            (а лучше -e '.[notebook]')
import sys, pathlib
ROOT = pathlib.Path.cwd()
# если ноутбук запущен из train/notebooks/ — добавим train/src в путь
src = (ROOT.parent / 'src') if ROOT.name == 'notebooks' else (ROOT / 'src')
if src.exists():
    sys.path.insert(0, str(src))

from uavtrain import datasets, prepare_visual, prepare_audio, train_visual, train_acoustic, evaluate, export
from uavtrain.config import ensure_dirs, DATA_DIR, RUNS_DIR, MODELS_DIR, PREPARED_DIR
ensure_dirs()
print('data:', DATA_DIR)
print('runs:', RUNS_DIR)
print('models:', MODELS_DIR)
print('prepared:', PREPARED_DIR)

## 1. Скачка датасетов

Реестр — `uavtrain.datasets.REGISTRY`. Для датасетов с прямой ссылкой `download(name)` качает + проверяет sha256 + распаковывает в `train/data/<name>/`. Для остальных — печатает инструкцию ручной загрузки.

In [ ]:
for spec in datasets.list_datasets():
    src = 'url' if spec.url else 'manual'
    print(f'{spec.name:24s} {spec.modality:7s} {spec.role:12s} [{src}]  {spec.description}')

In [ ]:
# Пример: автозагрузка ESC-50 (негативы для акустики).
# Остальные датасеты — по инструкции (см. spec.manual_instructions).
try:
    path = datasets.download('esc-50')
    print('esc-50 ->', path)
except Exception as e:
    print('esc-50:', e)

for name in ['dut-anti-uav', 'drone-vs-bird', 'multiclass-acoustic', 'drone-audio-dataset', 'mmaud']:
    print('\n---', name, '---')
    print(datasets.REGISTRY[name].manual_instructions)

## 2. Подготовка данных

**Видео** → YOLO-формат (`images/`, `labels/`, `data.yaml`) с train/val/test split по сцене/видео.
**Аудио** → окна MFCC/мел-спектрограмм + метки «дрон / не-дрон» (`features.npz`, `index.csv`, `meta.json`), split по исходному файлу.

_Парсеры конкретных датасетов и извлечение признаков пока — заготовки (поднимают `NotImplementedError` с подсказкой, что доделать под фактическую структуру архивов)._

In [ ]:
# Видео:
try:
    data_yaml = prepare_visual.convert(datasets=['drone-vs-bird'])  # или ['dut-anti-uav', 'drone-vs-bird']
    print('data.yaml:', data_yaml)
except NotImplementedError as e:
    print('TODO prepare_visual:', e)

In [ ]:
# Аудио:
try:
    features_npz = prepare_audio.build(positives=['multiclass-acoustic', 'drone-audio-dataset'], negatives=['esc-50'])
    print('features.npz:', features_npz)
except NotImplementedError as e:
    print('TODO prepare_audio:', e)

## 3. Обучение YOLOv8 (visual)

Дообучение `yolov8s` на едином UAV-датасете. Запускать на сервере с GPU. Артефакты — `train/runs/visual/<name>/` (Ultralytics).

In [ ]:
from uavtrain.train_visual import VisualTrainConfig
# best_pt = train_visual.train(VisualTrainConfig(data_yaml=data_yaml, epochs=100, batch=16, device='0'))
# print('best:', best_pt)
print('Раскомментируйте после подготовки data.yaml и на машине с GPU.')

## 4. Обучение lightweight CNN (acoustic)

Компактная CNN по MFCC/мел-спектрограммам «дрон / не-дрон». Артефакты — `train/runs/acoustic/<name>/`. _(Архитектура и тренировочный цикл — заготовка, реализуются с акустической веткой.)_

In [ ]:
from uavtrain.train_acoustic import AcousticTrainConfig
# best_lwcnn = train_acoustic.train(AcousticTrainConfig(features_npz=features_npz, epochs=50, device='cuda'))
# print('best:', best_lwcnn)
print('Раскомментируйте после реализации акустической ветки и подготовки features.npz.')

## 5. Оценка на test

Visual: mAP@0.5 / mAP@0.5:0.95, precision, recall (+ confusion matrix). Acoustic: accuracy/precision/recall/F1 (+ confusion matrix). Отчёты — `train/runs/eval/<name>/metrics.json` + png.

In [ ]:
# rep_v = evaluate.evaluate_visual(best_pt, data_yaml, device='0')
# print('visual metrics:', rep_v.metrics, '->', rep_v.artifacts_dir)
# rep_a = evaluate.evaluate_acoustic(best_lwcnn, features_npz)
# print('acoustic metrics:', rep_a.metrics, '->', rep_a.artifacts_dir)
print('Раскомментируйте после обучения соответствующих моделей.')

## 6. Export весов в `../models/`

Лучшие веса → `models/visual/yolov8s-uav.pt` и `models/acoustic/lwcnn.pt`; строка добавляется в `models/README.md` (реестр весов; сами файлы в git не коммитятся).

In [ ]:
# export.export_visual(best_pt, metrics_json=rep_v.artifacts_dir / 'metrics.json')
# export.export_acoustic(best_lwcnn, metrics_json=rep_a.artifacts_dir / 'metrics.json')
print('Раскомментируйте после eval. Дальше — поднять инфру и запустить пайплайн: см. корневой README (этапы 7-9).')